In [1]:
import numpy as np
from ruamel.yaml import YAML
import itertools
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Rectangle, Polygon
from IPython.display import clear_output
import torch
import sys
import pandas as pd
import tarfile
import math
import warnings
import os
import math
import csv
import gym
import argparse
import datetime
import pandas as pd
from IPython.display import clear_output, display
import json
import copy
# import torch.optim.lr_scheduler as scheduler
# from scipy.interpolate import RBFInterpolator
warnings.filterwarnings('ignore')
import settings
import torch.nn.functional as F
import glob
import yaml

In [2]:
def get_data_dir(subfolder):
    current_dir = os.getcwd()
    print(current_dir)
    return os.path.join(current_dir, "experiment_data", f"{subfolder}")


In [3]:
current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

In [4]:
def derived_papi(PAPI_data):
    DERIVED = {}
    DERIVED['TOT_INS_PER_CYC'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_TOT_INS']['instantaneous_value']) / np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_TOT_INS'].time)
    })
    DERIVED['TOT_CYC_PER_INS'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']) / np.array(PAPI_data['PAPI_TOT_INS']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_TOT_CYC'].time)
    })
    DERIVED['L3_TCM_PER_TCA'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_L3_TCM']['instantaneous_value']) / np.array(PAPI_data['PAPI_L3_TCA']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_L3_TCM'].time)
    })  
    DERIVED['TOT_STL_PER_CYC'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_RES_STL']['instantaneous_value']) / np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_RES_STL'].time)
    })
    
    # Create a mask for non-NaN values across all keys
    mask = ~DERIVED['TOT_INS_PER_CYC']['value'].isna()  # Start with one key to create the mask
    for key in DERIVED:
        mask &= ~DERIVED[key]['value'].isna()  # Combine masks for all keys

    for key in DERIVED:
        DERIVED[key] = DERIVED[key][mask]  # Filter each DataFrame using the combined mask

    return DERIVED

def calculate_power_with_wraparound(current, previous, time_diff, wraparound_value=262143.328850):
    diff = current - previous
    if diff < 0:  # Wraparound detected
        diff = (wraparound_value - previous) + current
    return diff / time_diff

def compute_measured_power(pubPower, all_data):
    PCAP_data = all_data['PCAP']
    first_PCAP_point = PCAP_data['timestamp'].iloc[0]
    new_pubPower = pubPower[pubPower['time'] >= first_PCAP_point] 
    elapsed_time = new_pubPower['time']-new_pubPower['time'].iloc[0]
    new_pubPower['elapsed_time'] = elapsed_time    
    new_pubPower.rename(columns={'time': 'timestamp'}, inplace=True)
    return new_pubPower

def compute_power(pubEnergy):
    power = {}
    geopm_sensor0 = geopm_sensor1 = pd.DataFrame({'timestamp':[],'value':[]})
    for i,row in pubEnergy.iterrows():
        if i%2 == 0:
            geopm_sensor0 = pd.concat([geopm_sensor0, pd.DataFrame([{'timestamp': row['time'], 'value': row['value']}])], ignore_index=True)
        else:
            geopm_sensor1 = pd.concat([geopm_sensor1, pd.DataFrame([{'timestamp': row['time'], 'value': row['value']}])], ignore_index=True)


    power['geopm_power_0'] = pd.DataFrame({
        'timestamp': geopm_sensor0['timestamp'][1:],  # Add timestamps
        'power': [
            calculate_power_with_wraparound(
                geopm_sensor0['value'][i],
                geopm_sensor0['value'][i-1],
                geopm_sensor0['timestamp'][i] - geopm_sensor0['timestamp'][i-1]
            ) for i in range(1, len(geopm_sensor0))
        ]
    })

    # Apply the same logic to geopm_power_1
    power['geopm_power_1'] = pd.DataFrame({
        'timestamp': geopm_sensor1['timestamp'][1:],  # Add timestamps
        'power': [
            calculate_power_with_wraparound(
                geopm_sensor1['value'][i],
                geopm_sensor1['value'][i-1],
                geopm_sensor1['timestamp'][i] - geopm_sensor1['timestamp'][i-1]
            ) for i in range(1, len(geopm_sensor1))
        ]
    })

    min_length = min(len(power['geopm_power_0']), len(power['geopm_power_1']))
    geopm_power_0 = power['geopm_power_0'][:min_length]
    geopm_power_1 = power['geopm_power_1'][:min_length]

    average_power = pd.DataFrame({
        'timestamp': geopm_power_0['timestamp'],  # Use the timestamp from geopm_power_0
        'average_power': [(p0 + p1) / 2 for p0, p1 in zip(geopm_power_0['power'], geopm_power_1['power'])]
    })
    average_power['elapsed_time'] = average_power['timestamp'] - average_power['timestamp'].iloc[0]
    power['average_power'] = average_power
    return power

def measure_progress(progress_data, all_data):
    power_data = all_data['measured_power']
    PCAP_data = all_data['PCAP']
    Progress_DATA = {} 
    progress_sensor = pd.DataFrame(progress_data)
    first_PCAP_point = PCAP_data['timestamp'].iloc[0]
    new_progress_sensor = progress_sensor[progress_sensor['time'] >= first_PCAP_point]
    new_power_data = power_data[power_data['timestamp'] >= first_PCAP_point]
    first_sensor_point = min(new_power_data['timestamp'].iloc[0], new_progress_sensor['time'].iloc[0])
    new_progress_sensor['elapsed_time'] = new_progress_sensor['time'] - first_sensor_point  
    performance_elapsed_time = new_progress_sensor.elapsed_time
    frequency_values = [
        progress_data['value'].iloc[t] / (performance_elapsed_time.iloc[t] - performance_elapsed_time.iloc[t-1]) for t in range(1, len(performance_elapsed_time))
    ]
    frequency_values = [0] + frequency_values  
    new_progress_sensor['frequency'] = frequency_values
    upsampled_timestamps= PCAP_data['timestamp']
    # progress_frequency_median = pd.DataFrame({'median': np.nanmedian(new_progress_sensor['frequency'].where(new_progress_sensor['time'] <= upsampled_timestamps.iloc[0])), 'timestamp': upsampled_timestamps.iloc[0]}, index=[0])
    progress_frequency_median = pd.DataFrame()
    for t in range(1, len(upsampled_timestamps)):
        progress_frequency_median = pd.concat([progress_frequency_median, pd.DataFrame({'median': [np.nanmedian(new_progress_sensor['frequency'].where((new_progress_sensor['time'] >= upsampled_timestamps.iloc[t-1]) & (new_progress_sensor['time'] <= upsampled_timestamps.iloc[t])))],
        'timestamp': [upsampled_timestamps.iloc[t]]})], ignore_index=True)
    progress_frequency_median['elapsed_time'] = progress_frequency_median['timestamp'] - progress_frequency_median['timestamp'].iloc[0]
    Progress_DATA['progress_sensor'] = new_progress_sensor
    Progress_DATA['progress_frequency_median'] = progress_frequency_median
    Progress_DATA['progress_sensor'].rename(columns={'time': 'timestamp'}, inplace=True)
    return Progress_DATA

def collect_papi(PAPI_data,all_data):
    PCAP_data = all_data['PCAP']
    first_PCAP_point = PCAP_data['timestamp'].iloc[0]
    new_PAPI_data = PAPI_data[PAPI_data['time'] >= first_PCAP_point]
    PAPI = {}
    for scope in new_PAPI_data['scope'].unique():
        scope_parts = scope.split('.')
        if len(scope_parts) > 4:  
            extracted_scope = scope_parts[3]
            PAPI[extracted_scope] = new_PAPI_data[new_PAPI_data['scope'] == scope]
            instantaneous_values = [0] + [PAPI[extracted_scope]['value'].iloc[k] - PAPI[extracted_scope]['value'].iloc[k-1] for k in range(1,len(PAPI[extracted_scope]))]
            PAPI[extracted_scope]['instantaneous_value'] = instantaneous_values
            PAPI[extracted_scope]['elapsed_time'] = PAPI[extracted_scope]['time'] - PAPI[extracted_scope]['time'].iloc[0]
    return PAPI


def generate_PCAP(PCAP_data):
    for row in PCAP_data.iterrows():
        if row[1]['time'] == 0:
            PCAP_data = PCAP_data.drop(row[0])


    PCAP_data['elapsed_time'] = PCAP_data['time'] - PCAP_data['time'].iloc[0]
    PCAP_data.rename(columns={'time': 'timestamp'}, inplace=True)
    return PCAP_data


In [5]:
DATA_DIR = get_data_dir("training_data")
root,folders,files = next(os.walk(DATA_DIR))
training_data = {}
MAX_PROGRESS = {}
for APP in folders:
    APP_DIR = os.path.join(DATA_DIR, APP)
    training_data[APP] = {}
    for file in next(os.walk(APP_DIR))[2]:
        training_data[APP][file] = {}
        if file.endswith('.tar'):
            tar_path = os.path.join(APP_DIR, file)
            extract_dir = os.path.join(APP_DIR, file[:-4])  
            
            if not os.path.exists(extract_dir):
                os.makedirs(extract_dir)
            
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=extract_dir)
            
        pubProgress = pd.read_csv(f'{extract_dir}/progress.csv')
        pubEnergy = pd.read_csv(f'{extract_dir}/energy.csv')
        pubPAPI = pd.read_csv(f'{extract_dir}/papi.csv')
        pubPCAP = pd.read_csv(f'{extract_dir}/PCAP_file.csv')
        pubPower = pd.read_csv(f'{extract_dir}/measured_power.csv')
        # training_data[APP][file]['power'] = compute_power(pubEnergy)
        training_data[APP][file]['PCAP'] = generate_PCAP(pubPCAP)
        training_data[APP][file]['measured_power'] = compute_measured_power(pubPower, training_data[APP][file])
        training_data[APP][file]['progress'] = measure_progress(pubProgress,training_data[APP][file])
        training_data[APP][file]['papi'] = collect_papi(pubPAPI,training_data[APP][file])
        training_data[APP][file]['derived_papi'] = derived_papi(training_data[APP][file]['papi'])   
    MAX_PROGRESS[APP] = training_data[APP][file]['progress']['progress_frequency_median']['median'].max()


/Users/akhileshraj/Documents/summer2024/main_codes


In [6]:
T_S = 1
ACTIONS = [78.0, 83.0, 89.0, 95.0, 101.0, 107.0, 112.0, 118.0, 124.0, 130.0, 136.0, 141.0, 147.0, 153.0, 159.0, 165.0]
exec_steps = 10000    
TOTAL_ACTIONS = len(ACTIONS)                                                                                                  # Total clock cycles needed for the execution of program.
ACTION_MIN = min(ACTIONS)                                                                                                    # Minima of control space
ACTION_MAX = max(ACTIONS)                                                                                                     # Maxima of control space
ACT_MID = ACTION_MIN + (ACTION_MAX - ACTION_MIN) / 2                                                                    # Midpoint of the control space to compute the normalized action space                     
OBS_MIN = np.zeros((5,))  
OBS_MAX = np.array([300,165,1,1,1])                                                                                 # Minima of observation space
OBS_MID = OBS_MIN + (OBS_MAX - OBS_MIN) / 2
EXEC_ITERATIONS = 10000
TOTAL_OBS = OBS_MAX - OBS_MIN
OBS_ONEHOT = 'onehot'
OBS_RANDOM = 'random'
OBS_SMOOTH = 'smooth'

def scale_reward_uniform(r, r_min=1.4666, r_max=7.7000, target_min=-10, target_max=1):
    return target_min + (r - r_min) * (target_max - target_min) / (r_max - r_min)


class SYS(object):
    def __init__(self,observation_type=OBS_ONEHOT,dim_obs=1,teps=0.0):
        super(SYS,self).__init__()

        self.num_actions = TOTAL_ACTIONS
        self.action_space = gym.spaces.Discrete(len(ACTIONS))  
        self.actions = ACTIONS  
        self.observation_space = gym.spaces.Box(low=OBS_MIN, high=OBS_MAX, shape=(5,), dtype=np.float32)  

    
    def reward(self, s, a, ns, measured_power, progress_track):
        """ 
        Returns the reward (float)
        """
        rho = 0.1
        if ns > 0:
            # reward_1 = -rho*(ns - progress_track)
            # reward_2 = -measured_power
            reward = np.array([ns, measured_power])
        else:
            reward = np.array([-100, -100])
        return reward
    
        


weighting_only = False
dataset_composition = 'random'
dataset_size = 1000
env_type = 'random'
env = SYS(observation_type=env_type, dim_obs=8, teps=0)

In [7]:
def get_roi_data(df, time_column, start_time, end_time):
    return df[(df[time_column] > start_time) & (df[time_column] <= end_time)]


In [8]:
def get_state(td, app, trace, start_time, end_time):
    ROI_progress = get_roi_data(td[app][trace]['progress']['progress_frequency_median'], 'timestamp', start_time, end_time)
    ROI_measured_power = get_roi_data(td[app][trace]['measured_power'], 'timestamp', start_time, end_time)
    TOT_INS_PER_CYC = get_roi_data(td[app][trace]['derived_papi']['TOT_INS_PER_CYC'], 'timestamp', start_time, end_time)
    L3_TCM_PER_TCA = get_roi_data(td[app][trace]['derived_papi']['L3_TCM_PER_TCA'], 'timestamp', start_time, end_time)
    TOT_STL_PER_CYC = get_roi_data(td[app][trace]['derived_papi']['TOT_STL_PER_CYC'], 'timestamp', start_time, end_time)
    return (
        ROI_progress['median'].mean() if not ROI_progress.empty else 0,
        ROI_measured_power['value'].mean() if not ROI_measured_power.empty else 0,
        TOT_INS_PER_CYC['value'].mean() if not TOT_INS_PER_CYC.empty else 0,
        L3_TCM_PER_TCA['value'].mean() if not L3_TCM_PER_TCA.empty else 0,
        TOT_STL_PER_CYC['value'].mean() if not TOT_STL_PER_CYC.empty else 0,
    )


In [9]:
training_dataset = []

for app in training_data:
    # if "ones-stream-full" in app:
    #     print(app)
    for trace in training_data[app]:
        pcap_data = training_data[app][trace]['PCAP']
        t1 = float('-inf')
        for i, row in pcap_data.iterrows():
            t2 = row['timestamp']
            state = get_state(training_data, app, trace, t1, t2)
            if i + 1 < len(pcap_data):
                t3 = pcap_data.iloc[i + 1]['timestamp']
                next_state = get_state(training_data, app, trace, t2, t3)
            else:
                next_state = state  # Use current state if it's the last row
            action = row['value']  # Assuming PCAP is in the 'value' column

            reward = env.reward(state[0], action, next_state[0], next_state[1], MAX_PROGRESS[app])
            
            # Add to training dataset
            if not np.isnan(next_state).any():
                training_dataset.append((app, state, action, reward, next_state))
            
            t1 = t2
print(len(training_dataset))
# build per-app mins/maxes from the assembled tuples
MAX_PROGRESS, MIN_PROGRESS = {}, {}
MAX_MPOWER,   MIN_MPOWER   = {}, {}

for app in training_data:
    prog = [entry[4][0] for entry in training_dataset if entry[0] == app]  # next_state progress
    powr = [entry[4][1] for entry in training_dataset if entry[0] == app]  # next_state measured_power
    MAX_PROGRESS[app], MIN_PROGRESS[app] = max(prog), min(prog)
    MAX_MPOWER[app],   MIN_MPOWER[app]   = max(powr), min(powr)

# Compute GLOBAL normalization bounds for states (used for all apps, including unknown ones)
GLOBAL_MIN_PROGRESS = min(MIN_PROGRESS.values())
GLOBAL_MAX_PROGRESS = max(MAX_PROGRESS.values())
GLOBAL_MIN_MPOWER = min(MIN_MPOWER.values())
GLOBAL_MAX_MPOWER = max(MAX_MPOWER.values())

print(f"Global state bounds - Progress: [{GLOBAL_MIN_PROGRESS:.2f}, {GLOBAL_MAX_PROGRESS:.2f}], Power: [{GLOBAL_MIN_MPOWER:.2f}, {GLOBAL_MAX_MPOWER:.2f}]")

# Save normalization bounds to a file
import pickle
bounds = {
    'MAX_PROGRESS': MAX_PROGRESS,
    'MIN_PROGRESS': MIN_PROGRESS,
    'MAX_MPOWER': MAX_MPOWER,
    'MIN_MPOWER': MIN_MPOWER,
    'GLOBAL_MIN_PROGRESS': GLOBAL_MIN_PROGRESS,
    'GLOBAL_MAX_PROGRESS': GLOBAL_MAX_PROGRESS,
    'GLOBAL_MIN_MPOWER': GLOBAL_MIN_MPOWER,
    'GLOBAL_MAX_MPOWER': GLOBAL_MAX_MPOWER
}
with open(os.path.join(DATA_DIR, 'normalization_bounds.pkl'), 'wb') as f:
    pickle.dump(bounds, f)
print("Normalization bounds saved to normalization_bounds.pkl")

def norm01(x, lo, hi):
    if hi <= lo: return 0.5
    x = max(min(x, hi), lo)
    return (x - lo) / (hi - lo)

normalized_training_dataset = []
for entry in training_dataset:
    app, state, action, reward_vec, next_state = entry
    # reward_vec = [measured_power, progress] per your SYS.reward
    prog, measured_power = float(reward_vec[0]), float(reward_vec[1])

    p_norm = norm01(measured_power, MIN_MPOWER[app], MAX_MPOWER[app])
    g_norm = norm01(prog,            MIN_PROGRESS[app], MAX_PROGRESS[app])

    # map to [-1,1] with similar magnitudes for stable training
    r2 = np.array([-(2*p_norm - 1),  (2*g_norm - 1)], dtype=np.float32)  # [-1,1] each
    normalized_training_dataset.append((app, state, action, r2, next_state))

# training_dataset = normalized_training_dataset

    
csv_file_name = 'training_dataset.csv'
csv_file_path = os.path.join(DATA_DIR, csv_file_name)

with open(csv_file_path, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    
    csv_writer.writerow(['App','Progress', 'Power', 'TOT_INS_PER_CYC', 'L3_TCM_PER_TCA', 'TOT_STL_PER_CYC', 
                         'Action', 'Reward', 
                         'Next_Progress', 'Next_Power', 'Next_TOT_INS_PER_CYC', 'Next_L3_TCM_PER_TCA', 'Next_STL_PER_CYC'])
    
    for app, state, action, reward, next_state in training_dataset:
        if not (np.isnan(state).any() or np.isnan(action) or np.isnan(next_state).any()): 
                # or (action-next_state[1]) < 0): 
                # or (action-next_state[1]) >= 10):
            state = np.array(state)
            next_state = np.array(next_state)
            if np.all((state >= 0) & (state <= 300)) and np.all((next_state >= 0) & (next_state <= 300)):
                row = [app] + list(state) + [action, reward] + list(next_state)
                csv_writer.writerow(row)



print(f"Training dataset has been saved to {csv_file_path}")


loaded_data = pd.read_csv(csv_file_path)
print(len(loaded_data))


csv_file_path = f'{DATA_DIR}/training_dataset.csv'

2032
Global state bounds - Progress: [4.50, 284.56], Power: [73.09, 167.11]
Normalization bounds saved to normalization_bounds.pkl
Training dataset has been saved to /Users/akhileshraj/Documents/summer2024/main_codes/experiment_data/training_data/training_dataset.csv
2029


In [10]:
# Data smoothing for preference-driven offline RL dataset
import pandas as pd
import numpy as np
from pathlib import Path

# Prefer DATA_DIR if defined; fall back to workspace-relative path
default_rel = Path("main_codes/experiment_data/training_data/training_dataset.csv")
data_path = Path(globals().get("DATA_DIR", "")) / "training_dataset.csv"
if not data_path.exists():
    data_path = default_rel
out_path = data_path.parent / "training_dataset_smoothed.csv"

df = pd.read_csv(data_path)

group_col = "App" if "App" in df.columns else None
exclude_cols = {"Action", "Reward"}
window = 5
ewm_alpha = 0.3

def smooth_group(g):
    g = g.copy()
    numeric_cols = g.select_dtypes(include=["number"]).columns
    numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
    if len(numeric_cols) == 0:
        return g
    # Rolling median for robustness, then EWMA for smooth trend
    g[numeric_cols] = g[numeric_cols].rolling(window=window, min_periods=1, center=True).median()
    g[numeric_cols] = g[numeric_cols].ewm(alpha=ewm_alpha, adjust=False).mean()
    return g

if group_col:
    df_smoothed = df.groupby(group_col, group_keys=False).apply(smooth_group)
else:
    df_smoothed = smooth_group(df)

df_smoothed.to_csv(out_path, index=False)
print(f"Saved smoothed dataset to: {out_path}")
df_smoothed.head()

Saved smoothed dataset to: /Users/akhileshraj/Documents/summer2024/main_codes/experiment_data/training_data/training_dataset_smoothed.csv


,App,Progress,Power,TOT_INS_PER_CYC,L3_TCM_PER_TCA,TOT_STL_PER_CYC,Action,Reward,Next_Progress,Next_Power,Next_TOT_INS_PER_CYC,Next_L3_TCM_PER_TCA,Next_STL_PER_CYC
0,ones-stream-add,197.867862,124.174798,0.143917,0.935807,0.879767,153.0,[205.44536059 145.48267834],197.867862,124.220718,0.148516,0.936419,0.879767
1,ones-stream-add,197.826265,124.181686,0.144607,0.935899,0.876272,124.0,[197.86786178 124.17479757],198.994930,127.410012,0.147867,0.936328,0.880045
2,ones-stream-add,197.838744,124.193395,0.144481,0.935871,0.877320,124.0,[197.59054991 124.22071787],200.910945,132.831812,0.146762,0.936171,0.880517
3,ones-stream-add,200.101615,130.580180,0.144392,0.935852,0.878610,147.0,[205.3816473 146.88234454],199.998020,130.248484,0.147288,0.936145,0.880292
4,ones-stream-add,199.431489,128.672342,0.145629,0.935922,0.878957,159.0,[205.63169723 147.89615067],199.275779,128.440154,0.147748,0.936127,0.879323


In [11]:
# Define the CSV file path
csv_file_path = f'{DATA_DIR}/training_dataset_smoothed.csv'

# Load the CSV into a DataFrame
loaded_data = pd.read_csv(csv_file_path)
print(len(loaded_data))

2029


In [12]:
# Convert the DataFrame back to the original format (list of tuples)
training_dataset_loaded = [
    (row[0],          # App
        tuple(row[1:6]),  # State
        row[6],          # Action
        row[7],          # Reward
        tuple(row[8:])   # Next State
    )
    for row in loaded_data.values
]

print(len(training_dataset_loaded))


2029


In [13]:
def load_coeff_files(folder):
    pattern = os.path.join(folder, 'inv_static_characteristics_*_coeffs.yaml')
    files = sorted(glob.glob(pattern))
    data = []
    for fp in files:
        try:
            with open(fp) as f:
                d = yaml.safe_load(f)
        except Exception as e:
            print(f"Skipping {fp}: failed to load YAML ({e})")
            continue
        if not d:
            print(f"Skipping {fp}: empty YAML")
            continue
        app = d.get('app') or os.path.basename(fp).replace('static_characteristics_','').replace('_coeffs.yaml','')
        coeffs = d.get('coefficients_perf')
        max_prog = d.get('max_progress', None)
        if coeffs is None:
            print(f"Skipping {fp}: no 'coefficients_perf' key")
            continue
        data.append({'path': fp, 'app': app, 'coeffs': coeffs, 'max_progress': max_prog})
    return data

def format_poly_str(coeffs):
    p = np.poly1d(coeffs)
    return str(p)

files_data = load_coeff_files('')
model_coeffs = {}
for item in files_data:
    model_coeffs[item['app']] = (item['coeffs'], item['max_progress'])

In [14]:
class FCNetwork(torch.nn.Module):
  def __init__(self, env, layers=[20,20]):
    super(FCNetwork, self).__init__()
    # self.all_observations = torch.tensor(stack_observations(env), dtype=torch.float32)
    dim_input = 6
    dim_output = env.num_actions * 2
    net_layers = []

    dim = dim_input
    for i, layer_size in enumerate(layers):
      net_layers.append(torch.nn.Linear(dim, layer_size))
      net_layers.append(torch.nn.ReLU())
      dim = layer_size
    net_layers.append(torch.nn.Linear(dim, dim_output))
    self.layers = net_layers
    self.network = torch.nn.Sequential(*net_layers)

  def forward(self, states):
    # observations = torch.index_select(self.all_observations, 0, states)
    states_tensor = torch.tensor(states, dtype=torch.float32)  # Ensure the correct dtype
    return self.network(states_tensor)

  def print_weights(self):
    for name, param in self.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.data.numpy()}")

In [15]:
state_means = np.array([150, 120, 0.5, 0.5, 0.5, 0.5])  # progress, power, 3 PAPI, pref
state_stds = np.array([90, 12, 0.15, 0.15, 0.15, 0.5])

def normalize_state(s):
    return (s - state_means) / state_stds
  

In [16]:
def calculate_weights(P_ref):
    """ 
    Returns the weights (float)
    """
    P_max = ACTION_MAX
    w1 = P_max/(P_max+P_ref)
    w2 = P_ref/(P_max+P_ref)
    weights = np.array([w1, w2])
    return weights

def translate_pre_perf_pow(coeffs, prog):
    poly = np.poly1d(coeffs)
    power = poly(prog)
    # ret_vector = calculate_weights(power)
    # ret_vector = np.array(ret_vector, dtype=np.float32)
    # x_unit = ret_vector
    x_unit = np.array([prog,power], dtype=np.float32)
    x_unit = x_unit / np.linalg.norm(x_unit)
    return x_unit

In [17]:
def get_tensors(list_of_tensors, list_of_indices, preference = None, **kwargs):
  s, a, ns, r, prefs, app, raw_s, raw_ns = [], [], [], [], [], [], [], []
  rho = 1

  for idx in list_of_indices:
    app_name = list_of_tensors[idx][0]
    app.append(app_name)
    
    # Normalize state using GLOBAL bounds: [progress, power, PAPI1, PAPI2, PAPI3]
    # This ensures consistent normalization for all apps (including unknown ones at eval time)
    state_raw = np.array(list_of_tensors[idx][1])
    raw_s.append(state_raw)
    state_norm = state_raw.copy()
    state_norm[0] = (state_raw[0] - GLOBAL_MIN_PROGRESS) / (GLOBAL_MAX_PROGRESS - GLOBAL_MIN_PROGRESS + 1e-8)
    state_norm[1] = (state_raw[1] - GLOBAL_MIN_MPOWER) / (GLOBAL_MAX_MPOWER - GLOBAL_MIN_MPOWER + 1e-8)
    # PAPI features [2:5] already in [0,1], keep as-is
    
    # Normalize next_state using GLOBAL bounds
    next_state_raw = np.array(list_of_tensors[idx][4])
    raw_ns.append(next_state_raw)
    next_state_norm = next_state_raw.copy()
    next_state_norm[0] = (next_state_raw[0] - GLOBAL_MIN_PROGRESS) / (GLOBAL_MAX_PROGRESS - GLOBAL_MIN_PROGRESS + 1e-8)
    next_state_norm[1] = (next_state_raw[1] - GLOBAL_MIN_MPOWER) / (GLOBAL_MAX_MPOWER - GLOBAL_MIN_MPOWER + 1e-8)
    
    # s.append(state_norm)
    s.append(state_raw)
    a.append(list_of_tensors[idx][2])
    # ns.append(next_state_norm)
    ns.append(next_state_raw)
    # index = np.random.choice(preference.shape[0])
    # pref_val = float(np.atleast_1d(preference[index])[0])
    
    # prefs.append([pref_val])
    
    # For reward: use APP-SPECIFIC normalization for accurate per-app targets
    progress_raw = next_state_raw[0]
    power_raw = next_state_raw[1]
    progress_norm_reward = (progress_raw - MIN_PROGRESS[app_name]) / (MAX_PROGRESS[app_name] - MIN_PROGRESS[app_name] + 1e-8)
    power_norm_reward = (power_raw - MIN_MPOWER[app_name]) / (MAX_MPOWER[app_name] - MIN_MPOWER[app_name] + 1e-8)
    
    prefs.append([progress_norm_reward])
    # Compute normalized progress target based on preference
    # progress_tar_norm = 1 - pref_val  # Target progress in [0,1] based on preference
    # Compute tracking error in normalized space using app-specific normalization
    # r_track_norm = rho * (progress_norm_reward - progress_tar_norm)
    r.append([progress_norm_reward, power_norm_reward]) 
    # Both rewards now in similar scale: tracking error ~[-1,1], power_norm ~[0,1]
    # Negate both to make higher values better (minimize tracking error, minimize power)
    # r.append([-r_track_norm, -power_norm])

  s = np.array(s)
  prefs_array = np.vstack(prefs)
  s_with_prefs = np.concatenate([s,prefs_array], axis=1)
  ns_with_prefs = np.concatenate([ns,prefs_array], axis=1)
  raw_s = np.array(raw_s)
  raw_ns = np.array(raw_ns)
  a = np.array(a)  
  r = np.array(r)
  return s_with_prefs, a, ns_with_prefs, r, prefs_array, s, ns, app,raw_s,raw_ns


In [18]:
def project_qvalues_cql_sampled(env, s, a, target_values, network, target_network,
                                optimizer, cql_alpha=0.1, num_steps=50, weights=None, preference=None):
    s = torch.tensor(s, dtype=torch.float32)
    a_idx = torch.tensor([ACTIONS.index(x) for x in a], dtype=torch.long)

    pred = network(s)                                  # [B, A*2]
    pred_vec = pred.view(pred.size(0), len(env.actions), 2)
    pred_sa = pred_vec[torch.arange(pred.size(0)), a_idx]   # [B,2]

    # Use MSE for better convergence near optimum
    loss = F.mse_loss(pred_sa, target_values)
    
    # Add gradient clipping for stability
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
    optimizer.step()

    tau = 0.005
    for p, tp in zip(network.parameters(), target_network.parameters()):
        tp.data.copy_(tau * p.data + (1 - tau) * tp.data)

    return float(loss.item())

In [19]:
def q_backup_sparse_sampled(env, t_network, app, s, a, ns, r, discount=0.99, preference=None, **kwargs):
    raw_s = kwargs.get('raw_s', None)
    raw_ns = kwargs.get('raw_ns', None)
    device = next(t_network.parameters()).device
    with torch.no_grad():
        target_q = t_network(ns).contiguous()                # [B, A*2]
    B, A = target_q.size(0), len(env.actions)

    # interpolators = kwargs['interpolators']
    models = kwargs['interpolators']
    pref_np = []
    pref_bmm = []
    for i,pre in enumerate(preference):
        p = float(np.atleast_1d(pre)[0])
        pref_np.append([translate_pre_perf_pow(models[app[i]][0], p*models[app[i]][1])])
        # pref_np.append(interpolators[app[i]]([pre]))
        pref_bmm.append(torch.from_numpy(np.array([1,1])).float().to(device))
    # pref_np = interpolators(preference)                     # [B,2] or [2]
    pref_np = np.vstack(pref_np)
    pref_bmm = np.vstack(pref_bmm)
    if pref_np.ndim == 1:
        pref_np = np.tile(pref_np, (B, 1))
        pref_bmm = np.tile(pref_bmm, (B, 1))
    prefs = torch.from_numpy(pref_np).to(device=device, dtype=torch.float32)  # [B,2]
    prefs_bmm = torch.from_numpy(pref_bmm).to(device=device, dtype=torch.float32)  # [B,2]

    q_flat = target_q.view(B, A, 2).reshape(B*A, 2)        # [B*A,2]
    prefs_flat = prefs.repeat_interleave(A, dim=0)         # [B*A,2]
    prefs_bmm_flat = prefs_bmm.repeat_interleave(A, dim=0)     # [B*A,2]
    
    cos = torch.clamp(F.cosine_similarity(prefs_flat, q_flat, dim=1), 0.0, 0.9999)  # [B*A]
    # dot = (prefs_bmm_flat * q_flat).sum(dim=1)                                          # [B*A]
    # dot = 1 
    dot = q_flat[:,0]/(1 + q_flat[:,1])  # Example: progress / (1 + power) to balance them
    # dot = 1
    score = (cos * dot).view(B, A)                                                  # [B,A]

    act = score.argmax(dim=1)                                    # [B]
    best_q_vec = target_q.view(B, A, 2)[torch.arange(B, device=device), act]   # [B,2]

    r_t = torch.as_tensor(r, dtype=torch.float32, device=device) # [B,2]
    return r_t + discount * best_q_vec                           # [B,2]

In [20]:
def conservative_q_iteration(env,
                             network, fig, axis,
                             num_itrs=100,
                             project_steps=20,
                             cql_alpha=0.1,
                             render=False,
                             weights=None,
                             sampled=False,
                             training_dataset=None,
                             log_file=None,learning_rate=1e-3,
                             preference = [0],
                             step_every=7000,
                             step_gamma=0.5,
                             batch_size=128,
                             **kwargs):
  """
  Runs Conservative Q-iteration.

  Args:
    env: A GridEnv object.
    num_itrs (int): Number of FQI iterations to run.
    project_steps (int): Number of gradient steps used for projection.
    cql_alpha (float): Value of weight on the CQL coefficient.
    render (bool): If True, will plot q-values after each iteration.
    sampled (bool): Whether to use sampled datasets for training or not.
    training_dataset (list): list of (s, a, r, ns) pairs
  """
  # print(log_file)
  # optimizer = torch.optim.Adam(network.parameters(), lr=learning_rate)
  optimizer = torch.optim.RMSprop(network.parameters(), lr=learning_rate)
#   scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.8, patience=3000, min_lr=1e-8, threshold=0.1)
#   # scheduler = scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2000, min_lr=1e-8, threshold=0.1, verbose=True)
#   scheduler = torch.optim.lr_scheduler.LambdaLR(
#     optimizer, lr_lambda=lambda step: (step_every/(step_every+step))**0.5
# )
  # scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9999)
# call scheduler.step() every iteration
    # Iteration-based LR scheduler
  scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=step_every, gamma=step_gamma
    )


  # q_values = np.zeros((dS, dA)) #Initializing the Q-values for getting the target values
  # q_values = network(s)
  loss_prev = 0
  target_network = copy.deepcopy(network)
  losses = []
  for i in range(num_itrs):
    epoch_losses = []
    for j in range(project_steps):
      # interpolators = kwargs.get('interpolators', None)
      models = kwargs.get('interpolators', None)
      training_idx = np.random.choice(np.arange(len(training_dataset)), size=batch_size)
      s, a, ns, r, prefs, _ , _, app, raw_s, raw_ns  = get_tensors(training_dataset, training_idx, preference)
      target_values = q_backup_sparse_sampled(env, target_network, app, s, a, ns, r, raw_s=raw_s, raw_ns=raw_ns, preference=prefs, **kwargs)
      new_prefs = []
      # for k,pre in enumerate(prefs):
      #   # new_prefs.append(models[app[k]]([pre]))
      #   new_prefs.append([translate_pre_perf_pow(models[app[k]][0], (1-pre[0])*models[app[k]][1])])
      # new_prefs = np.vstack(new_prefs)
      loss = project_qvalues_cql_sampled(
          env, s, a, target_values, network, target_network, optimizer,
          cql_alpha=cql_alpha, weights=None, preference=None
      )
      epoch_losses.append(loss)
      # scheduler.step(loss)
      if kwargs.get('adaptive_lr', True):
        scheduler.step()
    
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    current_lr = optimizer.param_groups[0]['lr']
    print(">"*10,current_lr) 
      # if j == project_steps - 1:
      #   q_values = intermed_values
    # Create a CSV writer object
      
      # log_file.writerow([i, np.mean(target_values), np.mean(target_values)])
    # interpolator = evaluate_interpolator(interpolator,kwargs['w_batch_interp'],kwargs['args'], env,state_for_eval,network)
    axis.plot(i, avg_loss, 'ro')
    axis.grid(True)
    print(f"Iter {i}: Loss = {avg_loss:.4f}")
    
    # Early stopping if converged
    if i > 50 and avg_loss < 0.003:
        print(f"Converged at iteration {i}")
        break
    # if np.mean([loss,loss_prev]) < 4.5:
    #   continue
    # else:
    #   loss_prev = loss
    # clear_output(wait=True)  # Clear the output to update the plot
    # display(fig)
  return target_network

In [21]:
def generate_w_batch_test(args, step_size):
    mesh_array = []
    step_size = step_size
    for i in range(args.reward_size):
        mesh_array.append(np.arange(0,1+step_size, step_size))
        
    w_batch_test = np.array(list(itertools.product(*mesh_array)))
    w_batch_test = w_batch_test[w_batch_test.sum(axis=1) == 1,:]
    w_batch_test = np.unique(w_batch_test,axis =0)
    
    return w_batch_test

In [22]:
# Initialize Interpolator
args = settings.HYPERPARAMS["HPC_MO_TD3_HER"]
args.reward_size = 2

# 16 points from (0,1) --> (1,0)
alphas = np.linspace(0.0, 1.0, 16)                      # 0, 1/15, ..., 1
# w_batch_line = np.column_stack([alphas, 1.0 - alphas])  # [[0,1], ..., [1,0]]

# (Optional) if you still need the dense mesh for other things:
# w_batch_test = generate_w_batch_test(args, step_size=args.w_step_size)

# Interpolator anchors
# interpolators = {}
# for file in next(os.walk('./'))[2]:
#     if 'interp' in file:
#         application_name = '_'.join(file.split('.')[0].split('_')[1:])
#         x = np.loadtxt(file, delimiter=",")
#         x_unit = x / np.linalg.norm(x, ord=2, axis=1, keepdims=True)
#         idx = np.round(np.linspace(0, len(w_batch_line) - 1, num=len(x))).astype(int)
#         w_batch_interp = w_batch_line[idx]
#         interp = RBFInterpolator(w_batch_interp, x_unit, kernel='linear')
#         interpolators[application_name] = interp
#         # break
# x = np.loadtxt("interp.txt", delimiter=",")
# x_unit = x / np.linalg.norm(x, ord=2, axis=1, keepdims=True)

# Match the number of anchors to x rows by sampling along the line
# idx = np.round(np.linspace(0, len(w_batch_line) - 1, num=len(x))).astype(int)
# w_batch_interp = w_batch_line

# interp = RBFInterpolator(w_batch_interp, x_unit, kernel='linear')

# print("16 line-sampled preferences (0,1 -> 1,0):\n", w_batch_line)


In [23]:
# model_coeffs

In [24]:
# # Initialize Interpolator
# args = settings.HYPERPARAMS["HPC_MO_TD3_HER"]
# args.reward_size = 2
# w_batch_test = generate_w_batch_test(args, step_size = args.w_step_size)
# # w_batch_eval = generate_w_batch_test(args, step_size = 0.005)
# w_batch_test_split = np.array_split(w_batch_test,16)
# x = np.loadtxt("interp.txt",delimiter=",")
# x_unit = x/np.linalg.norm(x,ord=2,axis=1,keepdims=True)
# idx_w_batch = np.round(np.linspace(0, len(w_batch_test)-1, num=len(x))).astype(int)
# w_batch_interp = w_batch_test[idx_w_batch]
# interp = RBFInterpolator(w_batch_interp, x_unit, kernel= 'linear')
# print(w_batch_test)
# print(w_batch_test_split)
# print(w_batch_interp)

In [ ]:
network = FCNetwork(env, layers=[50,20])
LR = 1e-3  # Lower learning rate for stability
cql_alpha_val = 0.01
weights = None
print (weighting_only)
plt.rcParams.update({'font.size': 12, "font.weight": "bold", 'axes.labelsize': 'x-large'})
fig,axs = plt.subplots(1,1,figsize=(10,5))
axs.set_xlabel('Iteration')
axs.set_ylabel('Loss value')
axs.set_title('RL actor network convergence')
plt.tight_layout()
csv_file_name = 'training_results.csv'
OUT_DIR = './trained_models'
output_file_path = os.path.join(OUT_DIR, csv_file_name)
preferences = alphas
# pref_pool = np.concatenate(preferences, axis=0) 
initial_state = {
       'model_state_dict': network.state_dict(),
       'epoch': 0,  # Starting epoch
   }

adaptive_lr  = True

with open(output_file_path, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    trained_net = conservative_q_iteration(env, network, fig, axs,
                                        num_itrs=1000, discount=0.95, cql_alpha=cql_alpha_val,
                                        weights=weights, render=False,
                                        sampled=not(weighting_only),
                                        training_dataset=training_dataset_loaded,log_file=csv_writer,learning_rate=LR,
                                        preference = preferences, interpolators = model_coeffs, w_batch_interp=None, 
                                        args=args, adaptive_lr=adaptive_lr, step_every=4000, step_gamma=0.5,
                                        batch_size=256)  # Larger batch for stability


torch.save(initial_state, f'./{OUT_DIR}/initial_state_all_preferences_{cql_alpha_val}_{LR}.pth')
fig.savefig(f'./{OUT_DIR}/Q-value_convergence.pdf')

False
>>>>>>>>>> 0.001
Iter 0: Loss = 21.3423
>>>>>>>>>> 0.001
Iter 1: Loss = 4.9465
>>>>>>>>>> 0.001
Iter 2: Loss = 4.1137
>>>>>>>>>> 0.001
Iter 3: Loss = 4.1573
>>>>>>>>>> 0.001
Iter 4: Loss = 3.7124
>>>>>>>>>> 0.001
Iter 5: Loss = 3.2076
>>>>>>>>>> 0.001
Iter 6: Loss = 3.9947
>>>>>>>>>> 0.001
Iter 7: Loss = 2.7757
>>>>>>>>>> 0.001
Iter 8: Loss = 2.0874
>>>>>>>>>> 0.001
Iter 9: Loss = 2.2269
>>>>>>>>>> 0.001
Iter 10: Loss = 1.5648
>>>>>>>>>> 0.001
Iter 11: Loss = 1.5752
>>>>>>>>>> 0.001
Iter 12: Loss = 1.4868
>>>>>>>>>> 0.001
Iter 13: Loss = 1.4268
>>>>>>>>>> 0.001
Iter 14: Loss = 1.5691
>>>>>>>>>> 0.001
Iter 15: Loss = 1.3923
>>>>>>>>>> 0.001
Iter 16: Loss = 1.1264
>>>>>>>>>> 0.001
Iter 17: Loss = 0.9405
>>>>>>>>>> 0.001
Iter 18: Loss = 0.7792
>>>>>>>>>> 0.001
Iter 19: Loss = 0.6793
>>>>>>>>>> 0.001
Iter 20: Loss = 0.6151
>>>>>>>>>> 0.001
Iter 21: Loss = 0.5993
>>>>>>>>>> 0.001
Iter 22: Loss = 0.5125
>>>>>>>>>> 0.001
Iter 23: Loss = 0.4236
>>>>>>>>>> 0.001
Iter 24: Loss = 0.3709
>>>

In [ ]:
def compress_results_folder(results_dir,current_time,cql_alpha,LR):
    tar_file_name = f'{results_dir}/training_results_{current_time}_all_preference_{cql_alpha}_{LR}.tar'
    
    # Create a tar file
    with tarfile.open(tar_file_name, 'w') as tar:
        # Iterate through the files in the results directory
        for item in os.listdir(results_dir):
            if item.endswith(('.pth', '.csv', '.pdf')):
                item_path = os.path.join(results_dir, item)
                # Check if it's a file (not a directory)
                # if os.path.isfile(item_path):
                tar.add(item_path, arcname=item)  # Add file to tar
                os.remove(item_path)

    print(f"Compressed files into {tar_file_name}")

# Specify the results directory


In [ ]:


torch.save(trained_net.state_dict(), f'{OUT_DIR}/trained_network_weights_{current_time}_all_preference_model_based_{cql_alpha_val}_{LR}.pth')  # Save the model weights
compress_results_folder(OUT_DIR,current_time,cql_alpha_val,LR)
print("Trained weights have been saved to 'trained_network_weights.pth'")



Compressed files into ./trained_models/training_results_20260211_223159_all_preference_0.01_0.001.tar
Trained weights have been saved to 'trained_network_weights.pth'
